# NB12 — ablation A7_window5: re-windowed feature cache + reference cells on the confirmation folds (CPU)

Re-windows the NB01 preprocessed recordings (only `preprocess.window_s` changes), rebuilds every feature and graph, then trains P3 x spatial and P3 x hybrid on the 25 confirmation splits, paired with their confirmation-grid runs.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[preprocess,dev]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_external.py tests/test_model_train.py', log=f'{WORK}/pytest_abl.txt')

In [ ]:
import glob
PRE = sorted(d for d in glob.glob('/kaggle/input/**/preproc', recursive=True) if glob.glob(f'{d}/sub-*.npz'))
PART = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
C = f'{PY} --config configs/ablation_a7_window5.yaml'
print(PRE, PART); assert len(PRE) == 3

In [ ]:
for k, d in enumerate(PRE):
    sh(f'{C} cache --preproc {d} --participants {PART} --out /tmp/cache_s{k}', log=f'{WORK}/cache_s{k}.log')
sh(f'{C} merge-caches --inputs /tmp/cache_s0 /tmp/cache_s1 /tmp/cache_s2 --out /tmp/cache')
sh(f'{C} qc --cache /tmp/cache --out {WORK}/qc_A7_window5.json --no-fail')

In [ ]:
sh(f'{C} grid --cache /tmp/cache --folds {SRC}/splits/folds_ds004504_confirm.json --pipelines P3 --edges spatial --tag A7_window5@P3xspatial --out {WORK}/ablations_confirm.jsonl --max-hours 5', log=f'{WORK}/A7_window5_spatial.log')
sh(f'{C} grid --cache /tmp/cache --folds {SRC}/splits/folds_ds004504_confirm.json --pipelines P3 --edges hybrid --tag A7_window5@P3xhybrid --out {WORK}/ablations_confirm.jsonl --max-hours 5', log=f'{WORK}/A7_window5_hybrid.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)